# Imports и базовая конфигурация

In [3]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

print("Thread limits set.")


Thread limits set.


In [4]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("SNA/data/processed")

MATCH_DATA_PARQUET = DATA_DIR / "match_dataset_model_ready.parquet"
MATCH_DATA_CSV = DATA_DIR / "match_dataset_model_ready.csv"

TRAIN_PATH = DATA_DIR / "train_matches_model_ready.parquet"
VAL_PATH = DATA_DIR / "val_matches_model_ready.parquet"
TEST_PATH = DATA_DIR / "test_matches_model_ready.parquet"

FEATURE_GROUPS_PATH = DATA_DIR / "feature_groups.json"
CLASS_WEIGHTS_PATH = DATA_DIR / "class_weights.json"

TEXT_EMB_PATH = DATA_DIR / "match_text_embeddings_allMiniLM.npy"
TEXT_EMB_INDEX_PATH = DATA_DIR / "match_text_embeddings_index.csv"

VAL_PRED_PATH = DATA_DIR / "model_predictions_val.csv"
TEST_PRED_PATH = DATA_DIR / "model_predictions_test.csv"
RESULTS_PATH = DATA_DIR / "model_results_ablation.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "target"

CLASS_NAMES = {
    0: "home_win",
    1: "draw",
    2: "away_win"
}

print("DATA_DIR:", DATA_DIR.resolve())


DATA_DIR: /Users/gaperov/Documents/University/SNA/data/processed


# Загрузка match_dataset_model_ready

In [5]:
def load_match_dataset():
    if MATCH_DATA_PARQUET.exists():
        print(f"Loading parquet: {MATCH_DATA_PARQUET}")
        df = pd.read_parquet(MATCH_DATA_PARQUET)
    elif MATCH_DATA_CSV.exists():
        print(f"Loading csv: {MATCH_DATA_CSV}")
        df = pd.read_csv(MATCH_DATA_CSV)
    else:
        raise FileNotFoundError(
            f"Neither {MATCH_DATA_PARQUET} nor {MATCH_DATA_CSV} exists."
        )
    return df


df_all = load_match_dataset()

print("df_all shape:", df_all.shape)
display(df_all.head())
print(df_all.columns.tolist())


Loading parquet: SNA/data/processed/match_dataset_model_ready.parquet
df_all shape: (500, 127)


,tournament_year,match_date,stage,home_team_name,away_team_name,home_team_norm,away_team_norm,home_score,away_score,stadium_name,...,fifa_strength_pc1,stage_norm,is_knockout,stage_final,stage_group,stage_quarter_final,stage_round_of_16,stage_semi_final,stage_third_place,no_text_flag
0,1994,1994-06-17,Group stage,Germany,Bolivia,germany,bolivia,1,0,"Soldier Field, Chicago",...,0.702889,group,0,0,1,0,0,0,0,1
1,1994,1994-06-17,Group stage,Spain,Korea Republic,spain,south korea,2,2,"Cotton Bowl, Dallas",...,0.511311,group,0,0,1,0,0,0,0,1
2,1994,1994-06-18,Group stage,Colombia,Romania,colombia,romania,1,3,"Rose Bowl, Los Angeles",...,-0.412544,group,0,0,1,0,0,0,0,1
3,1994,1994-06-18,Group stage,Italy,Republic of Ireland,italy,ireland,0,1,"Giants Stadium, New York/New Jersey",...,0.061937,group,0,0,1,0,0,0,0,1
4,1994,1994-06-18,Group stage,United States,Switzerland,united states,switzerland,1,1,"Pontiac Silverdome, Detroit",...,-0.514218,group,0,0,1,0,0,0,0,1


['tournament_year', 'match_date', 'stage', 'home_team_name', 'away_team_name', 'home_team_norm', 'away_team_norm', 'home_score', 'away_score', 'stadium_name', 'city', 'referee_name', 'attendance', 'home_manager_name', 'away_manager_name', 'tournament_id', 'match_id', 'home_team_id', 'away_team_id', 'stadium_id', 'referee_id', 'home_manager_id', 'away_manager_id', 'result', 'target', 'home_fifa_rank', 'home_fifa_points', 'home_fifa_previous_points', 'home_fifa_diff_points', 'home_fifa_ranking_date', 'away_fifa_rank', 'away_fifa_points', 'away_fifa_previous_points', 'away_fifa_diff_points', 'away_fifa_ranking_date', 'fifa_rank_diff', 'fifa_points_diff', 'fifa_momentum_diff', 'home_better_rank_flag', 'home_wc_matches_before', 'home_wc_points_before', 'home_wc_goals_for_before', 'home_wc_goals_against_before', 'home_wc_goal_diff_before', 'home_wc_wins_before', 'home_wc_win_rate_before', 'home_last3_matches', 'home_last3_points', 'home_last3_goals_for', 'home_last3_goals_against', 'home_las

# Базовая валидация колонок

In [6]:
required_cols = ["match_id", "tournament_year", TARGET_COL]

missing_required = [c for c in required_cols if c not in df_all.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df_all["tournament_year"] = df_all["tournament_year"].astype(int)
df_all[TARGET_COL] = df_all[TARGET_COL].astype(int)

print("Tournament years:", sorted(df_all["tournament_year"].unique()))
print("Target values:", sorted(df_all[TARGET_COL].unique()))


Tournament years: [np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]
Target values: [np.int64(0), np.int64(1), np.int64(2)]


# Загрузка или создание train / val / test split

In [7]:
def load_or_create_splits(df):
    if TRAIN_PATH.exists() and VAL_PATH.exists() and TEST_PATH.exists():
        print("Loading existing train/val/test parquet files.")
        train_df = pd.read_parquet(TRAIN_PATH)
        val_df = pd.read_parquet(VAL_PATH)
        test_df = pd.read_parquet(TEST_PATH)
    else:
        print("Split files not found. Creating temporal split by tournament_year.")
        
        train_years = [1994, 1998, 2002, 2006, 2010, 2014]
        val_years = [2018]
        test_years = [2022]
        
        train_df = df[df["tournament_year"].isin(train_years)].copy()
        val_df = df[df["tournament_year"].isin(val_years)].copy()
        test_df = df[df["tournament_year"].isin(test_years)].copy()
        
        train_df.to_parquet(TRAIN_PATH, index=False)
        val_df.to_parquet(VAL_PATH, index=False)
        test_df.to_parquet(TEST_PATH, index=False)
        
        print(f"Saved train to {TRAIN_PATH}")
        print(f"Saved val to {VAL_PATH}")
        print(f"Saved test to {TEST_PATH}")
    
    for d in [train_df, val_df, test_df]:
        d["tournament_year"] = d["tournament_year"].astype(int)
        d[TARGET_COL] = d[TARGET_COL].astype(int)
        
    return train_df, val_df, test_df


train_df, val_df, test_df = load_or_create_splits(df_all)

print("train shape:", train_df.shape)
print("val shape:", val_df.shape)
print("test shape:", test_df.shape)

print("train years:", sorted(train_df["tournament_year"].unique()))
print("val years:", sorted(val_df["tournament_year"].unique()))
print("test years:", sorted(test_df["tournament_year"].unique()))


Loading existing train/val/test parquet files.
train shape: (372, 127)
val shape: (64, 127)
test shape: (64, 127)
train years: [np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014)]
val years: [np.int64(2018)]
test years: [np.int64(2022)]


### Проверка target distribution

In [8]:
def show_target_distribution(df, name):
    vc = df[TARGET_COL].value_counts().sort_index()
    pct = df[TARGET_COL].value_counts(normalize=True).sort_index()
    
    out = pd.DataFrame({
        "count": vc,
        "share": pct
    })
    out.index = [f"{idx}_{CLASS_NAMES.get(idx, 'unknown')}" for idx in out.index]
    
    print(f"\nTarget distribution: {name}")
    display(out)


show_target_distribution(df_all, "all")
show_target_distribution(train_df, "train")
show_target_distribution(val_df, "val")
show_target_distribution(test_df, "test")



Target distribution: all


,count,share
0_home_win,219,0.438
1_draw,118,0.236
2_away_win,163,0.326



Target distribution: train


,count,share
0_home_win,164,0.440860
1_draw,90,0.241935
2_away_win,118,0.317204



Target distribution: val


,count,share
0_home_win,26,0.406250
1_draw,13,0.203125
2_away_win,25,0.390625



Target distribution: test


,count,share
0_home_win,29,0.453125
1_draw,15,0.234375
2_away_win,20,0.312500


### Проверка missing values

In [9]:
def missing_report(df, name, top_n=40):
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    print(f"\nMissing values: {name}")
    if len(miss) == 0:
        print("No missing values.")
    else:
        display(
            pd.DataFrame({
                "missing_count": miss,
                "missing_share": miss / len(df)
            }).head(top_n)
        )


missing_report(train_df, "train")
missing_report(val_df, "val")
missing_report(test_df, "test")



Missing values: train


,missing_count,missing_share
referee_name,255,0.685484
referee_id,255,0.685484
latest_text_time,151,0.405914
earliest_text_time,151,0.405914
away_fifa_ranking_date,9,0.024194
home_fifa_ranking_date,2,0.005376



Missing values: val


,missing_count,missing_share
latest_text_time,13,0.203125
earliest_text_time,13,0.203125



Missing values: test


,missing_count,missing_share
latest_text_time,15,0.234375
earliest_text_time,15,0.234375


### Загрузка feature_groups.json или восстановление групп

In [10]:
LEAKAGE_COLS = {
    "home_score",
    "away_score",
    "result",
    "target",
    "home_xg",
    "away_xg",
    "xg",
}

ID_META_COLS = {
    "match_id",
    "tournament_year",
    "home_team_name",
    "away_team_name",
    "match_date",
    "stage",
    "stage_norm",
    "combined_pre_match_text",
    "text_for_embedding",
    "sources",
    "latest_text_time",
    "earliest_text_time",
}


def is_numeric_col(df, col):
    return pd.api.types.is_numeric_dtype(df[col])


def filter_existing_numeric_features(df, features):
    features = [f for f in features if f in df.columns]
    features = [f for f in features if f not in LEAKAGE_COLS]
    features = [f for f in features if is_numeric_col(df, f)]
    return features


def infer_feature_groups(df):
    numeric_cols = [
        c for c in df.columns
        if is_numeric_col(df, c)
        and c not in LEAKAGE_COLS
        and c not in ["target"]
    ]
    
    groups = {}
    
    groups["ranking_full"] = [
        c for c in numeric_cols
        if (
            "fifa" in c.lower()
            or "rank" in c.lower()
            or "points_diff" in c.lower()
            or "momentum" in c.lower()
            or "better_rank" in c.lower()
        )
    ]
    
    groups["ranking_compact"] = [
        c for c in numeric_cols
        if c in [
            "fifa_rank_diff_home_minus_away",
            "fifa_rank_diff_away_minus_home",
            "fifa_points_diff_home_minus_away",
            "fifa_momentum_diff_home_minus_away",
            "fifa_strength_pc1",
            "fifa_rank_diff",
            "fifa_points_diff",
            "fifa_momentum_diff",
            "home_better_rank_flag",
        ]
    ]
    
    groups["form_last10"] = [
        c for c in numeric_cols
        if "last10" in c.lower()
    ]
    
    groups["form_last5"] = [
        c for c in numeric_cols
        if "last5" in c.lower()
    ]
    
    groups["form_last3"] = [
        c for c in numeric_cols
        if "last3" in c.lower()
    ]
    
    groups["wc_history"] = [
        c for c in numeric_cols
        if c.startswith("home_wc_") or c.startswith("away_wc_")
    ]
    
    groups["tournament_cumulative"] = [
        c for c in numeric_cols
        if "tournament" in c.lower() or "cum" in c.lower()
    ]
    
    groups["rest"] = [
        c for c in numeric_cols
        if "rest" in c.lower()
    ]
    
    groups["text_meta"] = [
        c for c in numeric_cols
        if c in [
            "text_available",
            "text_count",
            "log_text_count",
            "min_hours_before_match",
            "max_hours_before_match",
            "guardian_text_flag",
            "no_text_flag",
        ]
    ]
    
    groups["stage"] = [
        c for c in numeric_cols
        if c in [
            "is_knockout",
            "stage_group",
            "stage_round_of_16",
            "stage_quarter_final",
            "stage_semi_final",
            "stage_third_place",
            "stage_final",
        ]
    ]
    
    combined = []
    for g in [
        "ranking_full",
        "form_last10",
        "wc_history",
        "tournament_cumulative",
        "rest",
        "text_meta",
        "stage",
    ]:
        combined.extend(groups.get(g, []))
    
    groups["combined_numeric"] = sorted(list(dict.fromkeys(combined)))
    
    return groups


if FEATURE_GROUPS_PATH.exists():
    print(f"Loading feature groups from {FEATURE_GROUPS_PATH}")
    with open(FEATURE_GROUPS_PATH, "r", encoding="utf-8") as f:
        feature_groups = json.load(f)
else:
    print("feature_groups.json not found. Inferring feature groups from dataframe columns.")
    feature_groups = infer_feature_groups(df_all)
    
    with open(FEATURE_GROUPS_PATH, "w", encoding="utf-8") as f:
        json.dump(feature_groups, f, indent=2, ensure_ascii=False)
    
    print(f"Saved inferred feature groups to {FEATURE_GROUPS_PATH}")


# Clean feature groups: keep only existing numeric non-leakage columns
feature_groups_clean = {}
for group_name, feats in feature_groups.items():
    feature_groups_clean[group_name] = filter_existing_numeric_features(df_all, feats)

feature_groups = feature_groups_clean

print("Available feature groups:")
for group_name, feats in feature_groups.items():
    print(f"{group_name}: {len(feats)} features")
    print(feats[:20], "..." if len(feats) > 20 else "")


Loading feature groups from SNA/data/processed/feature_groups.json
Available feature groups:
ranking_full: 15 features
['home_fifa_rank', 'away_fifa_rank', 'fifa_rank_diff', 'home_fifa_points', 'away_fifa_points', 'fifa_points_diff', 'home_fifa_diff_points', 'away_fifa_diff_points', 'fifa_momentum_diff', 'home_better_rank_flag', 'fifa_rank_diff_home_minus_away', 'fifa_rank_diff_away_minus_home', 'fifa_points_diff_home_minus_away', 'fifa_momentum_diff_home_minus_away', 'fifa_strength_pc1'] 
ranking_compact: 3 features
['fifa_rank_diff_away_minus_home', 'home_better_rank_flag', 'fifa_strength_pc1'] 
form_last10: 12 features
['home_last10_matches', 'home_last10_points', 'home_last10_goals_for', 'home_last10_goals_against', 'home_last10_goal_diff', 'home_last10_win_rate', 'away_last10_matches', 'away_last10_points', 'away_last10_goals_for', 'away_last10_goals_against', 'away_last10_goal_diff', 'away_last10_win_rate'] 
form_last5: 12 features
['home_last5_matches', 'home_last5_points', 'hom

### Загрузка class_weights.json или fallback

In [11]:
default_class_weights = {
    0: 1.00,
    1: 1.86,
    2: 1.35,
}

if CLASS_WEIGHTS_PATH.exists():
    print(f"Loading class weights from {CLASS_WEIGHTS_PATH}")
    with open(CLASS_WEIGHTS_PATH, "r", encoding="utf-8") as f:
        loaded_weights = json.load(f)
    
    class_weights = {int(k): float(v) for k, v in loaded_weights['computed_balanced'].items()}
else:
    print("class_weights.json not found. Using default recommended class weights.")
    class_weights = default_class_weights
    
    with open(CLASS_WEIGHTS_PATH, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in class_weights.items()}, f, indent=2)
    
    print(f"Saved default class weights to {CLASS_WEIGHTS_PATH}")

print("class_weights:", class_weights)


Loading class weights from SNA/data/processed/class_weights.json
class_weights: {0: 0.76103500761035, 1: 1.4124293785310735, 2: 1.0224948875255624}


### Проверка text coverage

In [12]:
def ensure_text_columns(df):
    df = df.copy()
    
    if "text_available" not in df.columns:
        if "combined_pre_match_text" in df.columns:
            df["text_available"] = df["combined_pre_match_text"].fillna("").str.len().gt(0).astype(int)
        else:
            df["text_available"] = 0
    
    if "text_count" not in df.columns:
        df["text_count"] = df["text_available"].astype(int)
    
    if "log_text_count" not in df.columns:
        df["log_text_count"] = np.log1p(df["text_count"].fillna(0))
    
    return df


df_all = ensure_text_columns(df_all)
train_df = ensure_text_columns(train_df)
val_df = ensure_text_columns(val_df)
test_df = ensure_text_columns(test_df)


def text_coverage_report(df, name):
    print(f"\nText coverage: {name}")
    n = len(df)
    available = int(df["text_available"].fillna(0).sum())
    print(f"Rows: {n}")
    print(f"Text available: {available} / {n} = {available / n:.3f}")
    
    if "text_count" in df.columns:
        display(df["text_count"].describe())


text_coverage_report(df_all, "all")
text_coverage_report(train_df, "train")
text_coverage_report(val_df, "val")
text_coverage_report(test_df, "test")



Text coverage: all
Rows: 500
Text available: 321 / 500 = 0.642


count    500.000000
mean       1.686000
std        1.632401
min        0.000000
25%        0.000000
50%        1.000000
75%        3.000000
max        5.000000
Name: text_count, dtype: float64


Text coverage: train
Rows: 372
Text available: 221 / 372 = 0.594


count    372.000000
mean       1.591398
std        1.679023
min        0.000000
25%        0.000000
50%        1.000000
75%        3.000000
max        5.000000
Name: text_count, dtype: float64


Text coverage: val
Rows: 64
Text available: 51 / 64 = 0.797


count    64.000000
mean      1.812500
std       1.295597
min       0.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       5.000000
Name: text_count, dtype: float64


Text coverage: test
Rows: 64
Text available: 49 / 64 = 0.766


count    64.000000
mean      2.109375
std       1.604727
min       0.000000
25%       1.000000
50%       2.000000
75%       3.000000
max       5.000000
Name: text_count, dtype: float64

# Text embeddings

### Подготовка поля text_for_embedding

In [13]:
import re

def clean_text_for_embedding(text, max_chars=12000):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    
    if len(text) > max_chars:
        text = text[:max_chars]
    
    return text


def prepare_text_for_embedding(df):
    df = df.copy()
    
    if "text_for_embedding" not in df.columns:
        if "combined_pre_match_text" not in df.columns:
            print("combined_pre_match_text not found. Creating empty text_for_embedding.")
            df["text_for_embedding"] = ""
        else:
            df["text_for_embedding"] = df["combined_pre_match_text"].apply(clean_text_for_embedding)
    else:
        df["text_for_embedding"] = df["text_for_embedding"].fillna("").apply(clean_text_for_embedding)
    
    # Leakage-safe assumption: source table already filtered <= match_date - 6h
    df["text_for_embedding"] = df["text_for_embedding"].fillna("")
    
    return df


df_all = prepare_text_for_embedding(df_all)
train_df = prepare_text_for_embedding(train_df)
val_df = prepare_text_for_embedding(val_df)
test_df = prepare_text_for_embedding(test_df)

print(df_all[["match_id", "text_available", "text_for_embedding"]].tail())
print("Non-empty texts:", (df_all["text_for_embedding"].str.len() > 0).sum())


               match_id  text_available  \
495  match_eb04f9fd8d79               1   
496  match_f6887b5f5d6f               1   
497  match_ebf4a0fe598d               1   
498  match_c50654aea08c               1   
499  match_438f9a929e58               1   

                                    text_for_embedding  
495  World Cup 2022: England and France players fac...  
496  World Cup 2022: Scaloni defends Argentina beha...  
497  World Cup 2022: news and buildup to Argentina ...  
498  World Cup 2022 briefing: Argentina v France wi...  
499  World Cup 2022: Argentina and France train the...  
Non-empty texts: 321


### Установка sentence-transformers, если нужно

In [14]:
try:
    from sentence_transformers import SentenceTransformer
    print("sentence-transformers is available.")
except ImportError:
    print("sentence-transformers is not installed.")
    print("Install it in notebook with:")
    print("!pip install -U sentence-transformers")
    raise


sentence-transformers is available.


### Создание или загрузка text embeddings

In [15]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EXPECTED_EMB_DIM = 384

def create_or_load_text_embeddings(df):
    if TEXT_EMB_PATH.exists() and TEXT_EMB_INDEX_PATH.exists():
        print(f"Loading existing embeddings from {TEXT_EMB_PATH}")
        embeddings = np.load(TEXT_EMB_PATH)
        emb_index = pd.read_csv(TEXT_EMB_INDEX_PATH)
        
        print("Loaded embeddings shape:", embeddings.shape)
        print("Loaded index shape:", emb_index.shape)
        return embeddings, emb_index
    
    print(f"Creating embeddings with {EMBEDDING_MODEL_NAME}")
    
    df_emb = df.copy().reset_index(drop=True)
    df_emb["embedding_row"] = np.arange(len(df_emb))
    
    texts = df_emb["text_for_embedding"].fillna("").tolist()
    has_text_mask = np.array([len(t.strip()) > 0 for t in texts])
    
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    emb_dim = model.get_sentence_embedding_dimension()
    print("Embedding dimension:", emb_dim)
    
    embeddings = np.zeros((len(df_emb), emb_dim), dtype=np.float32)
    
    non_empty_texts = [texts[i] for i in np.where(has_text_mask)[0]]
    non_empty_indices = np.where(has_text_mask)[0]
    
    print(f"Encoding non-empty texts: {len(non_empty_texts)} / {len(texts)}")
    
    if len(non_empty_texts) > 0:
        non_empty_embeddings = model.encode(
            non_empty_texts,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=False
        ).astype(np.float32)
        
        embeddings[non_empty_indices] = non_empty_embeddings
    
    index_cols = [
        "match_id",
        "tournament_year",
        "home_team_name",
        "away_team_name",
        "text_available",
        "embedding_row",
    ]
    
    existing_index_cols = [c for c in index_cols if c in df_emb.columns]
    emb_index = df_emb[existing_index_cols].copy()
    
    if "embedding_row" not in emb_index.columns:
        emb_index["embedding_row"] = df_emb["embedding_row"]
    
    np.save(TEXT_EMB_PATH, embeddings)
    emb_index.to_csv(TEXT_EMB_INDEX_PATH, index=False)
    
    print(f"Saved embeddings to {TEXT_EMB_PATH}")
    print(f"Saved embeddings index to {TEXT_EMB_INDEX_PATH}")
    
    return embeddings, emb_index


text_embeddings_all, emb_index = create_or_load_text_embeddings(df_all)

print("text_embeddings_all shape:", text_embeddings_all.shape)
display(emb_index.head())


Loading existing embeddings from SNA/data/processed/match_text_embeddings_allMiniLM.npy
Loaded embeddings shape: (500, 384)
Loaded index shape: (500, 6)
text_embeddings_all shape: (500, 384)


,match_id,tournament_year,home_team_name,away_team_name,text_available,embedding_row
0,match_24ab401684d2,1994,Germany,Bolivia,0,0
1,match_38ef6f79f047,1994,Spain,Korea Republic,0,1
2,match_ebe85d2cc918,1994,Colombia,Romania,0,2
3,match_f0b250615ce7,1994,Italy,Republic of Ireland,0,3
4,match_fd7cb15508dc,1994,United States,Switzerland,0,4


### Проверка соответствия embeddings и индекса

In [16]:
assert len(emb_index) == text_embeddings_all.shape[0], "Index length != embeddings rows"
assert emb_index["embedding_row"].is_unique, "embedding_row is not unique"
assert emb_index["match_id"].is_unique, "match_id in embedding index is not unique"

print("Embedding index validation passed.")


Embedding index validation passed.


### Создание X_text_train / val / test по match_id

In [17]:
match_id_to_emb_row = dict(zip(emb_index["match_id"], emb_index["embedding_row"]))

def get_text_matrix_by_match_id(df, embeddings, match_id_to_row):
    rows = []
    missing = []
    
    for mid in df["match_id"].tolist():
        if mid not in match_id_to_row:
            missing.append(mid)
        else:
            rows.append(match_id_to_row[mid])
    
    if missing:
        raise ValueError(f"Missing embeddings for match_id: {missing[:10]} ... total={len(missing)}")
    
    return embeddings[np.array(rows)]


X_text_train = get_text_matrix_by_match_id(train_df, text_embeddings_all, match_id_to_emb_row)
X_text_val = get_text_matrix_by_match_id(val_df, text_embeddings_all, match_id_to_emb_row)
X_text_test = get_text_matrix_by_match_id(test_df, text_embeddings_all, match_id_to_emb_row)

y_train = train_df[TARGET_COL].astype(int).values
y_val = val_df[TARGET_COL].astype(int).values
y_test = test_df[TARGET_COL].astype(int).values

print("X_text_train:", X_text_train.shape)
print("X_text_val:", X_text_val.shape)
print("X_text_test:", X_text_test.shape)


X_text_train: (372, 384)
X_text_val: (64, 384)
X_text_test: (64, 384)


# Numeric feature matrices

### Подготовка numeric matrices для feature group

In [18]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def get_feature_list(group_name):
    if group_name not in feature_groups:
        raise ValueError(f"Feature group '{group_name}' not found. Available: {list(feature_groups.keys())}")
    
    feats = feature_groups[group_name]
    feats = filter_existing_numeric_features(df_all, feats)
    
    if len(feats) == 0:
        raise ValueError(f"Feature group '{group_name}' has zero usable numeric features.")
    
    return feats


def make_numeric_matrices(group_name, scale=True):
    features = get_feature_list(group_name)
    
    X_train_raw = train_df[features].copy()
    X_val_raw = val_df[features].copy()
    X_test_raw = test_df[features].copy()
    
    if scale:
        preprocessor = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
    else:
        preprocessor = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
    
    X_train = preprocessor.fit_transform(X_train_raw)
    X_val = preprocessor.transform(X_val_raw)
    X_test = preprocessor.transform(X_test_raw)
    
    return X_train, X_val, X_test, features, preprocessor


X_num_train, X_num_val, X_num_test, combined_numeric_features, numeric_preprocessor = make_numeric_matrices(
    "combined_numeric",
    scale=True
)

print("combined_numeric features:", len(combined_numeric_features))
print("X_num_train:", X_num_train.shape)
print("X_num_val:", X_num_val.shape)
print("X_num_test:", X_num_test.shape)
print(combined_numeric_features)


combined_numeric features: 69
X_num_train: (372, 69)
X_num_val: (64, 69)
X_num_test: (64, 69)
['away_fifa_diff_points', 'away_fifa_points', 'away_fifa_rank', 'away_first_match_in_tournament', 'away_last10_goal_diff', 'away_last10_goals_against', 'away_last10_goals_for', 'away_last10_matches', 'away_last10_points', 'away_last10_win_rate', 'away_rest_days', 'away_tournament_goal_diff_before', 'away_tournament_goals_against_before', 'away_tournament_goals_for_before', 'away_tournament_matches_before', 'away_tournament_points_before', 'away_wc_goal_diff_before', 'away_wc_goals_against_before', 'away_wc_goals_for_before', 'away_wc_matches_before', 'away_wc_points_before', 'away_wc_win_rate_before', 'away_wc_wins_before', 'fifa_momentum_diff', 'fifa_momentum_diff_home_minus_away', 'fifa_points_diff', 'fifa_points_diff_home_minus_away', 'fifa_rank_diff', 'fifa_rank_diff_away_minus_home', 'fifa_rank_diff_home_minus_away', 'fifa_strength_pc1', 'guardian_text_flag', 'home_better_rank_flag', 'hom

### Подготовка ablation numeric matrices

In [19]:
NUMERIC_GROUPS_FOR_ABLATION = [
    "ranking_full",
    "ranking_compact",
    "form_last10",
    "wc_history",
    "text_meta",
    "stage",
    "combined_numeric",
]

numeric_matrices = {}

for group in NUMERIC_GROUPS_FOR_ABLATION:
    try:
        Xtr, Xv, Xte, feats, prep = make_numeric_matrices(group, scale=True)
        numeric_matrices[group] = {
            "X_train": Xtr,
            "X_val": Xv,
            "X_test": Xte,
            "features": feats,
            "preprocessor": prep,
        }
        print(f"{group}: {Xtr.shape}, features={len(feats)}")
    except Exception as e:
        print(f"Skipping group={group}: {e}")


ranking_full: (372, 15), features=15
ranking_compact: (372, 3), features=3
form_last10: (372, 12), features=12
wc_history: (372, 14), features=14
text_meta: (372, 7), features=7
stage: (372, 7), features=7
combined_numeric: (372, 69), features=69


# Evaluation utilities

### Единая функция оценки моделей

In [20]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

def predict_proba_safe(model, X):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        
        # Ensure shape n x 3 and correct columns if model.classes_ is not [0,1,2]
        if hasattr(model, "classes_"):
            full_proba = np.zeros((X.shape[0], 3), dtype=float)
            for i, cls in enumerate(model.classes_):
                full_proba[:, int(cls)] = proba[:, i]
            return full_proba
        
        return proba
    
    # Fallback: one-hot from predictions
    pred = model.predict(X)
    proba = np.zeros((len(pred), 3), dtype=float)
    proba[np.arange(len(pred)), pred.astype(int)] = 1.0
    return proba


def evaluate_predictions(y_true, y_pred, y_proba=None, verbose=True, title=None):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        zero_division=0
    )
    
    metrics = {
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "home_win_precision": precision[0],
        "home_win_recall": recall[0],
        "home_win_f1": f1[0],
        "draw_precision": precision[1],
        "draw_recall": recall[1],
        "draw_f1": f1[1],
        "away_win_precision": precision[2],
        "away_win_recall": recall[2],
        "away_win_f1": f1[2],
    }
    
    if verbose:
        if title:
            print("\n" + "=" * 80)
            print(title)
            print("=" * 80)
        
        print("Accuracy:", acc)
        print("Balanced accuracy:", bal_acc)
        print("Macro F1:", macro_f1)
        print("Weighted F1:", weighted_f1)
        print("Draw recall:", recall[1])
        print("Draw F1:", f1[1])
        
        print("\nConfusion matrix:")
        print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))
        
        print("\nClassification report:")
        print(
            classification_report(
                y_true,
                y_pred,
                labels=[0, 1, 2],
                target_names=[CLASS_NAMES[i] for i in [0, 1, 2]],
                zero_division=0
            )
        )
    
    return metrics


def evaluate_model(model, X, y, verbose=True, title=None):
    y_pred = model.predict(X)
    y_proba = predict_proba_safe(model, X)
    metrics = evaluate_predictions(y, y_pred, y_proba, verbose=verbose, title=title)
    return metrics, y_pred, y_proba


### Results registry

In [21]:
results = []
trained_models = {}

def add_result(
    model_name,
    feature_set,
    val_metrics,
    test_metrics,
    model=None,
    extra=None
):
    row = {
        "model_name": model_name,
        "feature_set": feature_set,
        "accuracy_val": val_metrics["accuracy"],
        "balanced_accuracy_val": val_metrics["balanced_accuracy"],
        "macro_f1_val": val_metrics["macro_f1"],
        "weighted_f1_val": val_metrics["weighted_f1"],
        "draw_recall_val": val_metrics["draw_recall"],
        "draw_f1_val": val_metrics["draw_f1"],
        "accuracy_test": test_metrics["accuracy"],
        "balanced_accuracy_test": test_metrics["balanced_accuracy"],
        "macro_f1_test": test_metrics["macro_f1"],
        "weighted_f1_test": test_metrics["weighted_f1"],
        "draw_recall_test": test_metrics["draw_recall"],
        "draw_f1_test": test_metrics["draw_f1"],
    }
    
    if extra:
        row.update(extra)
    
    results.append(row)
    
    if model is not None:
        trained_models[model_name] = model


def get_results_df():
    return pd.DataFrame(results).sort_values("macro_f1_val", ascending=False)


# Baseline models

### Импорт моделей

In [22]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# Optional libraries
HAS_XGBOOST = False
HAS_LIGHTGBM = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
    print("XGBoost available.")
except ImportError:
    print("XGBoost not available. Optional block will be skipped.")

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
    print("LightGBM available.")
except ImportError:
    print("LightGBM not available. Optional block will be skipped.")


XGBoost available.
LightGBM available.


### Factory-функции моделей

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

def make_logreg():
    return LogisticRegression(
        max_iter=2000,
        class_weight=class_weights,
        random_state=RANDOM_STATE,
        solver="lbfgs"
    )


def make_rf_safe():
    return RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight=class_weights,
        random_state=RANDOM_STATE,
        n_jobs=1
    )


def make_gradient_boosting_safe():
    return GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE
    )


def make_hist_gradient_boosting_safe():
    # Не поддерживает class_weight во всех версиях sklearn, зато часто стабильнее и быстрее
    return HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.05,
        max_leaf_nodes=15,
        random_state=RANDOM_STATE
    )


def make_mlp_safe():
    return MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=5e-4,
        max_iter=500,
        early_stopping=False,
        random_state=RANDOM_STATE
    )


### Utility для обучения и оценки sklearn модели

In [24]:
def fit_eval_register_model(
    model,
    model_name,
    feature_set,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    sample_weight=None,
    verbose=False
):
    if sample_weight is not None:
        try:
            model.fit(X_train, y_train, sample_weight=sample_weight)
        except TypeError:
            print(f"{model_name}: sample_weight not supported, fitting without sample_weight.")
            model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train)
    
    val_metrics, val_pred, val_proba = evaluate_model(
        model,
        X_val,
        y_val,
        verbose=verbose,
        title=f"{model_name} | VAL"
    )
    
    test_metrics, test_pred, test_proba = evaluate_model(
        model,
        X_test,
        y_test,
        verbose=verbose,
        title=f"{model_name} | TEST"
    )
    
    add_result(
        model_name=model_name,
        feature_set=feature_set,
        val_metrics=val_metrics,
        test_metrics=test_metrics,
        model=model
    )
    
    return model, val_metrics, test_metrics


### Majority baseline

In [25]:
dummy = DummyClassifier(strategy="most_frequent")

# Для Dummy нужен любой X с правильным количеством строк
X_dummy_train = np.zeros((len(y_train), 1))
X_dummy_val = np.zeros((len(y_val), 1))
X_dummy_test = np.zeros((len(y_test), 1))

fit_eval_register_model(
    model=dummy,
    model_name="majority",
    feature_set="none",
    X_train=X_dummy_train,
    y_train=y_train,
    X_val=X_dummy_val,
    y_val=y_val,
    X_test=X_dummy_test,
    y_test=y_test,
    verbose=True
)

get_results_df()



majority | VAL
Accuracy: 0.40625
Balanced accuracy: 0.3333333333333333
Macro F1: 0.19259259259259257
Weighted F1: 0.2347222222222222
Draw recall: 0.0
Draw F1: 0.0

Confusion matrix:
[[26  0  0]
 [13  0  0]
 [25  0  0]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.41      1.00      0.58        26
        draw       0.00      0.00      0.00        13
    away_win       0.00      0.00      0.00        25

    accuracy                           0.41        64
   macro avg       0.14      0.33      0.19        64
weighted avg       0.17      0.41      0.23        64


majority | TEST
Accuracy: 0.453125
Balanced accuracy: 0.3333333333333333
Macro F1: 0.2078853046594982
Weighted F1: 0.2825940860215054
Draw recall: 0.0
Draw F1: 0.0

Confusion matrix:
[[29  0  0]
 [15  0  0]
 [20  0  0]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.45      1.00      0.62        29
        draw       0.

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
0,majority,none,0.40625,0.333333,0.192593,0.234722,0.0,0.0,0.453125,0.333333,0.207885,0.282594,0.0,0.0


### Ranking-only baseline

In [26]:
for ranking_group in ["ranking_compact", "ranking_full"]:
    if ranking_group not in numeric_matrices:
        print(f"Skipping {ranking_group}: not available.")
        continue
    
    Xtr = numeric_matrices[ranking_group]["X_train"]
    Xv = numeric_matrices[ranking_group]["X_val"]
    Xte = numeric_matrices[ranking_group]["X_test"]
    
    fit_eval_register_model(
        model=make_logreg(),
        model_name=f"{ranking_group}_logreg",
        feature_set=ranking_group,
        X_train=Xtr,
        y_train=y_train,
        X_val=Xv,
        y_val=y_val,
        X_test=Xte,
        y_test=y_test,
        verbose=True
    )
    
    fit_eval_register_model(
        model=make_rf_safe(),
        model_name=f"{ranking_group}_rf",
        feature_set=ranking_group,
        X_train=Xtr,
        y_train=y_train,
        X_val=Xv,
        y_val=y_val,
        X_test=Xte,
        y_test=y_test,
        verbose=True
    )

get_results_df()



ranking_compact_logreg | VAL
Accuracy: 0.484375
Balanced accuracy: 0.4430769230769231
Macro F1: 0.4515577507598784
Weighted F1: 0.5033214998100304
Draw recall: 0.23076923076923078
Draw F1: 0.1875

Confusion matrix:
[[14  7  5]
 [ 5  3  5]
 [ 2  9 14]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.67      0.54      0.60        26
        draw       0.16      0.23      0.19        13
    away_win       0.58      0.56      0.57        25

    accuracy                           0.48        64
   macro avg       0.47      0.44      0.45        64
weighted avg       0.53      0.48      0.50        64


ranking_compact_logreg | TEST
Accuracy: 0.515625
Balanced accuracy: 0.4954022988505747
Macro F1: 0.4936902904382579
Weighted F1: 0.5229672230434426
Draw recall: 0.4
Draw F1: 0.36363636363636365

Confusion matrix:
[[17  6  6]
 [ 4  6  5]
 [ 4  6 10]]

Classification report:
              precision    recall  f1-score   support

    home_win

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
0,majority,none,0.406250,0.333333,0.192593,0.234722,0.000000,0.000000,0.453125,0.333333,0.207885,0.282594,0.000000,0.000000


### Combined numeric baseline

#### debug. Проверка combined numeric matrices

In [27]:
Xtr = numeric_matrices["combined_numeric"]["X_train"]
Xv = numeric_matrices["combined_numeric"]["X_val"]
Xte = numeric_matrices["combined_numeric"]["X_test"]

print("Xtr shape:", Xtr.shape)
print("Xv shape:", Xv.shape)
print("Xte shape:", Xte.shape)

print("Xtr dtype:", Xtr.dtype)
print("Xv dtype:", Xv.dtype)
print("Xte dtype:", Xte.dtype)

print("Xtr NaN:", np.isnan(Xtr).sum())
print("Xv NaN:", np.isnan(Xv).sum())
print("Xte NaN:", np.isnan(Xte).sum())

print("Xtr inf:", np.isinf(Xtr).sum())
print("Xv inf:", np.isinf(Xv).sum())
print("Xte inf:", np.isinf(Xte).sum())

print("Xtr min/max:", np.nanmin(Xtr), np.nanmax(Xtr))
print("Xv min/max:", np.nanmin(Xv), np.nanmax(Xv))
print("Xte min/max:", np.nanmin(Xte), np.nanmax(Xte))

print("y_train distribution:")
print(pd.Series(y_train).value_counts().sort_index())


Xtr shape: (372, 69)
Xv shape: (64, 69)
Xte shape: (64, 69)
Xtr dtype: float64
Xv dtype: float64
Xte dtype: float64
Xtr NaN: 0
Xv NaN: 0
Xte NaN: 0
Xtr inf: 0
Xv inf: 0
Xte inf: 0
Xtr min/max: -4.183551652345433 7.810249675906653
Xv min/max: -3.8574754514753873 7.810249675906653
Xte min/max: -3.1996544136447214 7.810249675906653
y_train distribution:
0    164
1     90
2    118
Name: count, dtype: int64


### Combined numeric LogisticRegression only

In [28]:
model = make_logreg()

fit_eval_register_model(
    model=model,
    model_name="combined_numeric_logreg",
    feature_set="combined_numeric",
    X_train=Xtr,
    y_train=y_train,
    X_val=Xv,
    y_val=y_val,
    X_test=Xte,
    y_test=y_test,
    verbose=True
)

get_results_df()



combined_numeric_logreg | VAL
Accuracy: 0.546875
Balanced accuracy: 0.4728205128205129
Macro F1: 0.4425469267574531
Weighted F1: 0.5073955649613544
Draw recall: 0.07692307692307693
Draw F1: 0.10526315789473684

Confusion matrix:
[[12  4 10]
 [ 4  1  8]
 [ 2  1 22]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.67      0.46      0.55        26
        draw       0.17      0.08      0.11        13
    away_win       0.55      0.88      0.68        25

    accuracy                           0.55        64
   macro avg       0.46      0.47      0.44        64
weighted avg       0.52      0.55      0.51        64


combined_numeric_logreg | TEST
Accuracy: 0.4375
Balanced accuracy: 0.4157088122605364
Macro F1: 0.3903881392637015
Weighted F1: 0.4263376124437781
Draw recall: 0.13333333333333333
Draw F1: 0.16666666666666666

Confusion matrix:
[[12  3 14]
 [ 3  2 10]
 [ 2  4 14]]

Classification report:
              precision    recall  f1-

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667
0,majority,none,0.406250,0.333333,0.192593,0.234722,0.000000,0.000000,0.453125,0.333333,0.207885,0.282594,0.000000,0.000000


### Combined numeric RandomForest safe

In [29]:
model = make_rf_safe()

fit_eval_register_model(
    model=model,
    model_name="combined_numeric_rf_safe",
    feature_set="combined_numeric",
    X_train=Xtr,
    y_train=y_train,
    X_val=Xv,
    y_val=y_val,
    X_test=Xte,
    y_test=y_test,
    verbose=True
)

get_results_df()



combined_numeric_rf_safe | VAL
Accuracy: 0.53125
Balanced accuracy: 0.4830769230769231
Macro F1: 0.4898722543883835
Weighted F1: 0.5475009477529639
Draw recall: 0.23076923076923078
Draw F1: 0.1935483870967742

Confusion matrix:
[[14  7  5]
 [ 5  3  5]
 [ 0  8 17]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.74      0.54      0.62        26
        draw       0.17      0.23      0.19        13
    away_win       0.63      0.68      0.65        25

    accuracy                           0.53        64
   macro avg       0.51      0.48      0.49        64
weighted avg       0.58      0.53      0.55        64


combined_numeric_rf_safe | TEST
Accuracy: 0.546875
Balanced accuracy: 0.550191570881226
Macro F1: 0.5389355742296918
Weighted F1: 0.5522584033613445
Draw recall: 0.5333333333333333
Draw F1: 0.45714285714285713

Confusion matrix:
[[15  8  6]
 [ 3  8  4]
 [ 4  4 12]]

Classification report:
              precision    recall  f1-

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667
0,majority,none,0.406250,0.333333,0.192593,0.234722,0.000000,0.000000,0.453125,0.333333,0.207885,0.282594,0.000000,0.000000


### Combined numeric GradientBoosting safe

In [30]:
model = make_gradient_boosting_safe()

fit_eval_register_model(
    model=model,
    model_name="combined_numeric_gb_safe",
    feature_set="combined_numeric",
    X_train=Xtr,
    y_train=y_train,
    X_val=Xv,
    y_val=y_val,
    X_test=Xte,
    y_test=y_test,
    verbose=True
)

get_results_df()



combined_numeric_gb_safe | VAL
Accuracy: 0.546875
Balanced accuracy: 0.48256410256410254
Macro F1: 0.4805031446540881
Weighted F1: 0.5431132075471699
Draw recall: 0.15384615384615385
Draw F1: 0.16

Confusion matrix:
[[17  5  4]
 [ 6  2  5]
 [ 4  5 16]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.63      0.65      0.64        26
        draw       0.17      0.15      0.16        13
    away_win       0.64      0.64      0.64        25

    accuracy                           0.55        64
   macro avg       0.48      0.48      0.48        64
weighted avg       0.54      0.55      0.54        64


combined_numeric_gb_safe | TEST
Accuracy: 0.546875
Balanced accuracy: 0.5183908045977011
Macro F1: 0.517921146953405
Weighted F1: 0.5490591397849462
Draw recall: 0.4
Draw F1: 0.3870967741935484

Confusion matrix:
[[19  5  5]
 [ 4  6  5]
 [ 5  5 10]]

Classification report:
              precision    recall  f1-score   support

    home_wi

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667
0,majority,none,0.406250,0.333333,0.192593,0.234722,0.000000,0.000000,0.453125,0.333333,0.207885,0.282594,0.000000,0.000000


### Combined numeric HistGradientBoosting safe

In [31]:
model = make_hist_gradient_boosting_safe()

fit_eval_register_model(
    model=model,
    model_name="combined_numeric_histgb_safe",
    feature_set="combined_numeric",
    X_train=Xtr,
    y_train=y_train,
    X_val=Xv,
    y_val=y_val,
    X_test=Xte,
    y_test=y_test,
    verbose=True
)

get_results_df()



combined_numeric_histgb_safe | VAL
Accuracy: 0.5625
Balanced accuracy: 0.4707692307692308
Macro F1: 0.452991452991453
Weighted F1: 0.5412660256410255
Draw recall: 0.0
Draw F1: 0.0

Confusion matrix:
[[18  4  4]
 [ 8  0  5]
 [ 2  5 18]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.64      0.69      0.67        26
        draw       0.00      0.00      0.00        13
    away_win       0.67      0.72      0.69        25

    accuracy                           0.56        64
   macro avg       0.44      0.47      0.45        64
weighted avg       0.52      0.56      0.54        64


combined_numeric_histgb_safe | TEST
Accuracy: 0.5625
Balanced accuracy: 0.5132183908045977
Macro F1: 0.5017713365539452
Weighted F1: 0.543780193236715
Draw recall: 0.2
Draw F1: 0.2608695652173913

Confusion matrix:
[[20  3  6]
 [ 6  3  6]
 [ 5  2 13]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.65   

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667
0,majority,none,0.406250,0.333333,0.192593,0.234722,0.000000,0.000000,0.453125,0.333333,0.207885,0.282594,0.000000,0.000000


### Combined numeric MLP safe

In [32]:
model = make_mlp_safe()

fit_eval_register_model(
    model=model,
    model_name="combined_numeric_mlp_safe",
    feature_set="combined_numeric",
    X_train=Xtr,
    y_train=y_train,
    X_val=Xv,
    y_val=y_val,
    X_test=Xte,
    y_test=y_test,
    verbose=True
)

get_results_df()



combined_numeric_mlp_safe | VAL
Accuracy: 0.578125
Balanced accuracy: 0.49743589743589745
Macro F1: 0.4705882352941176
Weighted F1: 0.5392156862745098
Draw recall: 0.07692307692307693
Draw F1: 0.11764705882352941

Confusion matrix:
[[16  2  8]
 [ 5  1  7]
 [ 4  1 20]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.64      0.62      0.63        26
        draw       0.25      0.08      0.12        13
    away_win       0.57      0.80      0.67        25

    accuracy                           0.58        64
   macro avg       0.49      0.50      0.47        64
weighted avg       0.53      0.58      0.54        64


combined_numeric_mlp_safe | TEST
Accuracy: 0.40625
Balanced accuracy: 0.36647509578544063
Macro F1: 0.34276729559748426
Weighted F1: 0.3878242924528302
Draw recall: 0.06666666666666667
Draw F1: 0.1

Confusion matrix:
[[14  2 13]
 [ 3  1 11]
 [ 7  2 11]]

Classification report:
              precision    recall  f1-score   

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
9,combined_numeric_mlp_safe,combined_numeric,0.578125,0.497436,0.470588,0.539216,0.076923,0.117647,0.406250,0.366475,0.342767,0.387824,0.066667,0.100000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667
0,majority,none,0.406250,0.333333,0.192593,0.234722,0.000000,0.000000,0.453125,0.333333,0.207885,0.282594,0.000000,0.000000


### XGB and LGBM

In [33]:
def make_xgb():
    return XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


def make_lgbm():
    return LGBMClassifier(
        objective="multiclass",
        num_class=3,
        n_estimators=300,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=15,
        class_weight=class_weights,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

In [34]:
if HAS_XGBOOST:
    # sample_weight for class imbalance
    xgb_sample_weight = np.array([class_weights[int(y)] for y in y_train])
    
    fit_eval_register_model(
        model=make_xgb(),
        model_name="xgboost_combined_numeric",
        feature_set="combined_numeric",
        X_train=Xtr,
        y_train=y_train,
        X_val=Xv,
        y_val=y_val,
        X_test=Xte,
        y_test=y_test,
        sample_weight=xgb_sample_weight,
        verbose=True
    )


xgboost_combined_numeric | VAL
Accuracy: 0.515625
Balanced accuracy: 0.4692307692307692
Macro F1: 0.4791666666666667
Weighted F1: 0.5361328125
Draw recall: 0.23076923076923078
Draw F1: 0.1875

Confusion matrix:
[[15  8  3]
 [ 5  3  5]
 [ 2  8 15]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.68      0.58      0.62        26
        draw       0.16      0.23      0.19        13
    away_win       0.65      0.60      0.62        25

    accuracy                           0.52        64
   macro avg       0.50      0.47      0.48        64
weighted avg       0.56      0.52      0.54        64


xgboost_combined_numeric | TEST
Accuracy: 0.5625
Balanced accuracy: 0.5298850574712644
Macro F1: 0.530586401714397
Weighted F1: 0.5604361484511982
Draw recall: 0.4
Draw F1: 0.41379310344827586

Confusion matrix:
[[20  3  6]
 [ 5  6  4]
 [ 5  5 10]]

Classification report:
              precision    recall  f1-score   support

    home_win     

In [36]:
get_results_df()

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
10,xgboost_combined_numeric,combined_numeric,0.515625,0.469231,0.479167,0.536133,0.230769,0.187500,0.562500,0.529885,0.530586,0.560436,0.400000,0.413793
9,combined_numeric_mlp_safe,combined_numeric,0.578125,0.497436,0.470588,0.539216,0.076923,0.117647,0.406250,0.366475,0.342767,0.387824,0.066667,0.100000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667


# Numeric ablation: отдельные feature groups

In [35]:
for group in ["form_last10", "wc_history", "text_meta", "stage"]:
    if group not in numeric_matrices:
        print(f"Skipping {group}: not available.")
        continue
    
    Xtr = numeric_matrices[group]["X_train"]
    Xv = numeric_matrices[group]["X_val"]
    Xte = numeric_matrices[group]["X_test"]
    
    fit_eval_register_model(
        model=make_logreg(),
        model_name=f"{group}_logreg",
        feature_set=group,
        X_train=Xtr,
        y_train=y_train,
        X_val=Xv,
        y_val=y_val,
        X_test=Xte,
        y_test=y_test,
        verbose=True
    )

get_results_df()



form_last10_logreg | VAL
Accuracy: 0.375
Balanced accuracy: 0.3779487179487179
Macro F1: 0.3687565116136544
Weighted F1: 0.3934058420665564
Draw recall: 0.38461538461538464
Draw F1: 0.23809523809523808

Confusion matrix:
[[ 7 13  6]
 [ 2  5  6]
 [ 2 11 12]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.64      0.27      0.38        26
        draw       0.17      0.38      0.24        13
    away_win       0.50      0.48      0.49        25

    accuracy                           0.38        64
   macro avg       0.44      0.38      0.37        64
weighted avg       0.49      0.38      0.39        64


form_last10_logreg | TEST
Accuracy: 0.265625
Balanced accuracy: 0.2745210727969349
Macro F1: 0.27065238996482677
Weighted F1: 0.2793938451167976
Draw recall: 0.26666666666666666
Draw F1: 0.17391304347826086

Confusion matrix:
[[ 6 17  6]
 [ 5  4  6]
 [ 3 10  7]]

Classification report:
              precision    recall  f1-score   su

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
10,xgboost_combined_numeric,combined_numeric,0.515625,0.469231,0.479167,0.536133,0.230769,0.187500,0.562500,0.529885,0.530586,0.560436,0.400000,0.413793
9,combined_numeric_mlp_safe,combined_numeric,0.578125,0.497436,0.470588,0.539216,0.076923,0.117647,0.406250,0.366475,0.342767,0.387824,0.066667,0.100000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667


# Text-only baseline

### Text-only на всех матчах

In [37]:
text_only_models = [
    ("text_only_logreg_all_matches", make_logreg()),
    ("text_only_rf_all_matches", make_rf_safe()),
    ("text_only_mlp_all_matches", make_mlp_safe()),
]

for model_name, model in text_only_models:
    fit_eval_register_model(
        model=model,
        model_name=model_name,
        feature_set="text_embeddings_all_matches_zero_missing",
        X_train=X_text_train,
        y_train=y_train,
        X_val=X_text_val,
        y_val=y_val,
        X_test=X_text_test,
        y_test=y_test,
        verbose=True
    )

get_results_df()



text_only_logreg_all_matches | VAL
Accuracy: 0.328125
Balanced accuracy: 0.3123076923076923
Macro F1: 0.3135600907029478
Weighted F1: 0.32766156462585033
Draw recall: 0.23076923076923078
Draw F1: 0.24

Confusion matrix:
[[ 9  3 14]
 [ 9  3  1]
 [10  6  9]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.32      0.35      0.33        26
        draw       0.25      0.23      0.24        13
    away_win       0.38      0.36      0.37        25

    accuracy                           0.33        64
   macro avg       0.32      0.31      0.31        64
weighted avg       0.33      0.33      0.33        64


text_only_logreg_all_matches | TEST
Accuracy: 0.453125
Balanced accuracy: 0.4346743295019157
Macro F1: 0.41185506460648774
Weighted F1: 0.42970656004337215
Draw recall: 0.5333333333333333
Draw F1: 0.47058823529411764

Confusion matrix:
[[18  4  7]
 [ 6  8  1]
 [10  7  3]]

Classification report:
              precision    recall  f1-s

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
10,xgboost_combined_numeric,combined_numeric,0.515625,0.469231,0.479167,0.536133,0.230769,0.187500,0.562500,0.529885,0.530586,0.560436,0.400000,0.413793
9,combined_numeric_mlp_safe,combined_numeric,0.578125,0.497436,0.470588,0.539216,0.076923,0.117647,0.406250,0.366475,0.342767,0.387824,0.066667,0.100000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667


### Text-only обучение только на матчах с text_available = 1

In [39]:
train_text_mask = train_df["text_available"].fillna(0).astype(int).values == 1

print("Train with text:", train_text_mask.sum(), "/", len(train_text_mask))

X_text_train_available = X_text_train[train_text_mask]
y_train_available = y_train[train_text_mask]

text_only_available_models = [
    ("text_only_logreg_train_text_available_eval_all", make_logreg()),
    ("text_only_rf_train_text_available_eval_all", make_rf_safe()),
    ("text_only_mlp_train_text_available_eval_all", make_mlp_safe()),
]

for model_name, model in text_only_available_models:
    fit_eval_register_model(
        model=model,
        model_name=model_name,
        feature_set="text_embeddings_train_text_available_eval_all",
        X_train=X_text_train_available,
        y_train=y_train_available,
        X_val=X_text_val,
        y_val=y_val,
        X_test=X_text_test,
        y_test=y_test,
        verbose=True
    )

get_results_df()


Train with text: 221 / 372

text_only_logreg_train_text_available_eval_all | VAL
Accuracy: 0.359375
Balanced accuracy: 0.341025641025641
Macro F1: 0.32172008547008546
Weighted F1: 0.33602213541666665
Draw recall: 0.23076923076923078
Draw F1: 0.24

Confusion matrix:
[[ 5  3 18]
 [ 4  3  6]
 [ 4  6 15]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.38      0.19      0.26        26
        draw       0.25      0.23      0.24        13
    away_win       0.38      0.60      0.47        25

    accuracy                           0.36        64
   macro avg       0.34      0.34      0.32        64
weighted avg       0.36      0.36      0.34        64


text_only_logreg_train_text_available_eval_all | TEST
Accuracy: 0.40625
Balanced accuracy: 0.42088122605363987
Macro F1: 0.41108979823813585
Weighted F1: 0.41308510496589945
Draw recall: 0.5333333333333333
Draw F1: 0.47058823529411764

Confusion matrix:
[[11  4 14]
 [ 2  8  5]
 [ 6  7  7]]


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
10,xgboost_combined_numeric,combined_numeric,0.515625,0.469231,0.479167,0.536133,0.230769,0.187500,0.562500,0.529885,0.530586,0.560436,0.400000,0.413793
9,combined_numeric_mlp_safe,combined_numeric,0.578125,0.497436,0.470588,0.539216,0.076923,0.117647,0.406250,0.366475,0.342767,0.387824,0.066667,0.100000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636
5,combined_numeric_logreg,combined_numeric,0.546875,0.472821,0.442547,0.507396,0.076923,0.105263,0.437500,0.415709,0.390388,0.426338,0.133333,0.166667


# Numeric + Text baseline

### Создание fused matrices

In [40]:
def get_text_meta_for_fusion(df):
    cols = []
    
    for c in ["text_available", "log_text_count"]:
        if c in df.columns:
            cols.append(c)
    
    if not cols:
        return np.zeros((len(df), 0), dtype=np.float32), []
    
    arr = df[cols].fillna(0).astype(float).values
    return arr, cols


X_text_meta_train, fusion_text_meta_cols = get_text_meta_for_fusion(train_df)
X_text_meta_val, _ = get_text_meta_for_fusion(val_df)
X_text_meta_test, _ = get_text_meta_for_fusion(test_df)

X_fused_train = np.hstack([X_num_train, X_text_train, X_text_meta_train])
X_fused_val = np.hstack([X_num_val, X_text_val, X_text_meta_val])
X_fused_test = np.hstack([X_num_test, X_text_test, X_text_meta_test])

print("fusion text meta cols:", fusion_text_meta_cols)
print("X_fused_train:", X_fused_train.shape)
print("X_fused_val:", X_fused_val.shape)
print("X_fused_test:", X_fused_test.shape)


fusion text meta cols: ['text_available', 'log_text_count']
X_fused_train: (372, 455)
X_fused_val: (64, 455)
X_fused_test: (64, 455)


### Numeric + Text models

In [42]:
fused_models = [
    ("numeric_text_logreg", make_logreg()),
    ("numeric_text_rf", make_rf_safe()),
    ("numeric_text_mlp", make_mlp_safe()),
]

for model_name, model in fused_models:
    fit_eval_register_model(
        model=model,
        model_name=model_name,
        feature_set="combined_numeric_plus_text_embeddings",
        X_train=X_fused_train,
        y_train=y_train,
        X_val=X_fused_val,
        y_val=y_val,
        X_test=X_fused_test,
        y_test=y_test,
        verbose=True
    )

get_results_df()



numeric_text_logreg | VAL
Accuracy: 0.53125
Balanced accuracy: 0.4471794871794872
Macro F1: 0.40307971014492755
Weighted F1: 0.48051120923913043
Draw recall: 0.0
Draw F1: 0.0

Confusion matrix:
[[12  4 10]
 [ 6  0  7]
 [ 2  1 22]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.60      0.46      0.52        26
        draw       0.00      0.00      0.00        13
    away_win       0.56      0.88      0.69        25

    accuracy                           0.53        64
   macro avg       0.39      0.45      0.40        64
weighted avg       0.46      0.53      0.48        64


numeric_text_logreg | TEST
Accuracy: 0.4375
Balanced accuracy: 0.4157088122605364
Macro F1: 0.3903881392637015
Weighted F1: 0.4263376124437781
Draw recall: 0.13333333333333333
Draw F1: 0.16666666666666666

Confusion matrix:
[[12  4 13]
 [ 2  2 11]
 [ 3  3 14]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.7

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
22,numeric_text_rf,combined_numeric_plus_text_embeddings,0.640625,0.562051,0.546268,0.606456,0.153846,0.235294,0.578125,0.551724,0.553846,0.576923,0.400000,0.461538
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
10,xgboost_combined_numeric,combined_numeric,0.515625,0.469231,0.479167,0.536133,0.230769,0.187500,0.562500,0.529885,0.530586,0.560436,0.400000,0.413793
9,combined_numeric_mlp_safe,combined_numeric,0.578125,0.497436,0.470588,0.539216,0.076923,0.117647,0.406250,0.366475,0.342767,0.387824,0.066667,0.100000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636


# Ablation study

### Формирование ablation results table

In [43]:
results_df = get_results_df()

display(results_df)

results_df.to_csv(RESULTS_PATH, index=False)
print(f"Saved results to {RESULTS_PATH}")


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
22,numeric_text_rf,combined_numeric_plus_text_embeddings,0.640625,0.562051,0.546268,0.606456,0.153846,0.235294,0.578125,0.551724,0.553846,0.576923,0.400000,0.461538
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
10,xgboost_combined_numeric,combined_numeric,0.515625,0.469231,0.479167,0.536133,0.230769,0.187500,0.562500,0.529885,0.530586,0.560436,0.400000,0.413793
9,combined_numeric_mlp_safe,combined_numeric,0.578125,0.497436,0.470588,0.539216,0.076923,0.117647,0.406250,0.366475,0.342767,0.387824,0.066667,0.100000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636


Saved results to SNA/data/processed/model_results_ablation.csv


### Укороченная ablation table по ожидаемым моделям

In [44]:
expected_model_order = [
    "majority",
    "ranking_compact_logreg",
    "ranking_full_logreg",
    "ranking_full_rf",
    "form_last10_logreg",
    "wc_history_logreg",
    "combined_numeric_logreg",
    "combined_numeric_rf",
    "text_only_logreg_all_matches",
    "text_only_mlp_all_matches",
    "numeric_text_logreg",
    "numeric_text_mlp",
    "xgboost_combined_numeric",
    "lightgbm_combined_numeric",
]

ablation_short = results_df[results_df["model_name"].isin(expected_model_order)].copy()

ablation_short["model_name"] = pd.Categorical(
    ablation_short["model_name"],
    categories=expected_model_order,
    ordered=True
)

ablation_short = ablation_short.sort_values("model_name")

display(
    ablation_short[
        [
            "model_name",
            "feature_set",
            "accuracy_val",
            "macro_f1_val",
            "draw_recall_val",
            "draw_f1_val",
            "accuracy_test",
            "macro_f1_test",
            "draw_recall_test",
            "draw_f1_test",
        ]
    ]
)


,model_name,feature_set,accuracy_val,macro_f1_val,draw_recall_val,draw_f1_val,accuracy_test,macro_f1_test,draw_recall_test,draw_f1_test
0,majority,none,0.406250,0.192593,0.000000,0.000000,0.453125,0.207885,0.000000,0.000000
1,ranking_compact_logreg,ranking_compact,0.484375,0.451558,0.230769,0.187500,0.515625,0.493690,0.400000,0.363636
3,ranking_full_logreg,ranking_full,0.531250,0.463509,0.153846,0.160000,0.484375,0.467174,0.333333,0.303030
4,ranking_full_rf,ranking_full,0.531250,0.480545,0.153846,0.133333,0.500000,0.492308,0.533333,0.400000
11,form_last10_logreg,form_last10,0.375000,0.368757,0.384615,0.238095,0.265625,0.270652,0.266667,0.173913
12,wc_history_logreg,wc_history,0.406250,0.377630,0.153846,0.114286,0.468750,0.453531,0.400000,0.333333
5,combined_numeric_logreg,combined_numeric,0.546875,0.442547,0.076923,0.105263,0.437500,0.390388,0.133333,0.166667
15,text_only_logreg_all_matches,text_embeddings_all_matches_zero_missing,0.328125,0.313560,0.230769,0.240000,0.453125,0.411855,0.533333,0.470588
17,text_only_mlp_all_matches,text_embeddings_all_matches_zero_missing,0.390625,0.353843,0.153846,0.200000,0.453125,0.410837,0.466667,0.482759
21,numeric_text_logreg,combined_numeric_plus_text_embeddings,0.531250,0.403080,0.000000,0.000000,0.437500,0.390388,0.133333,0.166667


# Model selection и сохранение predictions

### Выбор лучшей модели по validation macro-F1

In [45]:
results_df = get_results_df()

best_row = results_df.iloc[0]
best_model_name = best_row["model_name"]

print("Best model by validation macro-F1:")
display(best_row)

best_model = trained_models[best_model_name]

print("Selected model:", best_model_name)


Best model by validation macro-F1:


model_name                ranking_compact_rf
feature_set                  ranking_compact
accuracy_val                         0.59375
balanced_accuracy_val               0.573846
macro_f1_val                        0.565187
weighted_f1_val                     0.596776
draw_recall_val                     0.461538
draw_f1_val                              0.4
accuracy_test                            0.5
balanced_accuracy_test              0.477969
macro_f1_test                       0.469039
weighted_f1_test                    0.497771
draw_recall_test                    0.266667
draw_f1_test                        0.275862
Name: 2, dtype: object

Selected model: ranking_compact_rf


### Получение правильных matrices для выбранной модели

In [46]:
def get_matrices_for_feature_set(feature_set):
    if feature_set == "none":
        return X_dummy_val, X_dummy_test
    
    if feature_set in numeric_matrices:
        return numeric_matrices[feature_set]["X_val"], numeric_matrices[feature_set]["X_test"]
    
    if feature_set == "combined_numeric":
        return X_num_val, X_num_test
    
    if feature_set == "text_embeddings_all_matches_zero_missing":
        return X_text_val, X_text_test
    
    if feature_set == "text_embeddings_train_text_available_eval_all":
        return X_text_val, X_text_test
    
    if feature_set == "combined_numeric_plus_text_embeddings":
        return X_fused_val, X_fused_test
    
    raise ValueError(f"Unknown feature_set: {feature_set}")


best_feature_set = best_row["feature_set"]
X_best_val, X_best_test = get_matrices_for_feature_set(best_feature_set)

print("Best feature set:", best_feature_set)
print("X_best_val:", X_best_val.shape)
print("X_best_test:", X_best_test.shape)


Best feature set: ranking_compact
X_best_val: (64, 3)
X_best_test: (64, 3)


### Финальная оценка лучшей модели на val и test

In [47]:
best_val_metrics, best_val_pred, best_val_proba = evaluate_model(
    best_model,
    X_best_val,
    y_val,
    verbose=True,
    title=f"BEST MODEL {best_model_name} | VAL 2018"
)

best_test_metrics, best_test_pred, best_test_proba = evaluate_model(
    best_model,
    X_best_test,
    y_test,
    verbose=True,
    title=f"BEST MODEL {best_model_name} | TEST 2022"
)



BEST MODEL ranking_compact_rf | VAL 2018
Accuracy: 0.59375
Balanced accuracy: 0.5738461538461538
Macro F1: 0.5651867512332629
Weighted F1: 0.5967758985200846
Draw recall: 0.46153846153846156
Draw F1: 0.4

Confusion matrix:
[[13  7  6]
 [ 2  6  5]
 [ 2  4 19]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.76      0.50      0.60        26
        draw       0.35      0.46      0.40        13
    away_win       0.63      0.76      0.69        25

    accuracy                           0.59        64
   macro avg       0.58      0.57      0.57        64
weighted avg       0.63      0.59      0.60        64


BEST MODEL ranking_compact_rf | TEST 2022
Accuracy: 0.5
Balanced accuracy: 0.4779693486590039
Macro F1: 0.46903906537297385
Weighted F1: 0.49777145625300556
Draw recall: 0.26666666666666666
Draw F1: 0.27586206896551724

Confusion matrix:
[[15  8  6]
 [ 4  4  7]
 [ 5  2 13]]

Classification report:
              precision    recall 

### Сохранение predictions для val/test

In [48]:
def make_predictions_df(df, y_true, y_pred, y_proba, model_name):
    out = pd.DataFrame()
    
    for col in ["match_id", "tournament_year", "home_team_name", "away_team_name"]:
        if col in df.columns:
            out[col] = df[col].values
        else:
            out[col] = None
    
    out["true_target"] = y_true.astype(int)
    out["pred_target"] = y_pred.astype(int)
    
    out["proba_home_win"] = y_proba[:, 0]
    out["proba_draw"] = y_proba[:, 1]
    out["proba_away_win"] = y_proba[:, 2]
    
    out["model_name"] = model_name
    
    return out


val_predictions_df = make_predictions_df(
    val_df,
    y_val,
    best_val_pred,
    best_val_proba,
    best_model_name
)

test_predictions_df = make_predictions_df(
    test_df,
    y_test,
    best_test_pred,
    best_test_proba,
    best_model_name
)

val_predictions_df.to_csv(VAL_PRED_PATH, index=False)
test_predictions_df.to_csv(TEST_PRED_PATH, index=False)

print(f"Saved val predictions to {VAL_PRED_PATH}")
print(f"Saved test predictions to {TEST_PRED_PATH}")

display(val_predictions_df.head())
display(test_predictions_df.head())


Saved val predictions to SNA/data/processed/model_predictions_val.csv
Saved test predictions to SNA/data/processed/model_predictions_test.csv


,match_id,tournament_year,home_team_name,away_team_name,true_target,pred_target,proba_home_win,proba_draw,proba_away_win,model_name
0,match_3ee0a05214bf,2018,Russia,Saudi Arabia,0,0,0.620546,0.290168,0.089286,ranking_compact_rf
1,match_0a7804398319,2018,Portugal,Spain,1,1,0.219522,0.613728,0.166749,ranking_compact_rf
2,match_53a59c1e0e89,2018,Morocco,IR Iran,2,2,0.064700,0.440879,0.494421,ranking_compact_rf
3,match_b0d45a81a705,2018,Egypt,Uruguay,2,2,0.034423,0.319926,0.645651,ranking_compact_rf
4,match_3bcb591a98d4,2018,Argentina,Iceland,1,1,0.363731,0.571607,0.064661,ranking_compact_rf


,match_id,tournament_year,home_team_name,away_team_name,true_target,pred_target,proba_home_win,proba_draw,proba_away_win,model_name
0,match_eba59d5d01d0,2022,Qatar,Ecuador,2,2,0.161103,0.383179,0.455718,ranking_compact_rf
1,match_0a8f5c4a4766,2022,United States,Wales,1,1,0.307446,0.499180,0.193374,ranking_compact_rf
2,match_80a4ca985d2c,2022,Senegal,Netherlands,2,2,0.067461,0.298156,0.634384,ranking_compact_rf
3,match_a64cba8917b3,2022,England,IR Iran,0,0,0.552648,0.358092,0.089260,ranking_compact_rf
4,match_281c5a33aca4,2022,Denmark,Tunisia,1,0,0.696415,0.234501,0.069084,ranking_compact_rf


# Optional graph baseline

### Загрузка graph nodes / edges

In [49]:
GRAPH_NODES_PATH = DATA_DIR / "graph_nodes_model_ready.parquet"
GRAPH_EDGES_PATH = DATA_DIR / "graph_edges_model_ready.parquet"

if GRAPH_NODES_PATH.exists() and GRAPH_EDGES_PATH.exists():
    graph_nodes = pd.read_parquet(GRAPH_NODES_PATH)
    graph_edges = pd.read_parquet(GRAPH_EDGES_PATH)
    
    print("graph_nodes:", graph_nodes.shape)
    print("graph_edges:", graph_edges.shape)
    
    display(graph_nodes.head())
    display(graph_edges.head())
else:
    print("Graph files not found. Skipping graph baseline.")


graph_nodes: (1007, 7)
graph_edges: (4741, 10)


,node_id,node_type,name,canonical_name,source_id,tournament_year,metadata_json
0,team_28ef36b35ae8,Team,Germany,germany,NaN,NaN,"{""team_norm"": ""germany""}"
1,team_1747d4b2dd4b,Team,Spain,spain,NaN,NaN,"{""team_norm"": ""spain""}"
2,team_25446782e2cc,Team,Colombia,colombia,NaN,NaN,"{""team_norm"": ""colombia""}"
3,team_bab4ac375ada,Team,Italy,italy,NaN,NaN,"{""team_norm"": ""italy""}"
4,team_152649df347e,Team,United States,united states,NaN,NaN,"{""team_norm"": ""united states""}"


,edge_id,source_node_id,target_node_id,edge_type,match_id,tournament_id,tournament_year,timestamp,weight,metadata_json
0,edge_8cdaebc5d11a,match_24ab401684d2,wc_1994,match_belongs_to_tournament,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,{}
1,edge_c8b436f81868,match_24ab401684d2,stadium_21a36eff12d1,match_played_at_stadium,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,{}
2,edge_e7376ce96f9e,referee_082a13461fbe,match_24ab401684d2,referee_officiated_match,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,{}
3,edge_cee57db5b986,team_28ef36b35ae8,match_24ab401684d2,team_played_match,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,"{""side"": ""home"", ""team_fifa_rank"": 3.0, ""team_..."
4,edge_bf0ca135aaf5,team_28ef36b35ae8,team_76781be1c79d,team_played_against_team,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,"{""side"": ""home"", ""stage_norm"": ""group"", ""is_kn..."


### Построение NetworkX graph

In [51]:
import networkx as nx

if GRAPH_NODES_PATH.exists() and GRAPH_EDGES_PATH.exists():
    G = nx.Graph()
    
    # Infer node id column
    possible_node_id_cols = [
        "node_id",
        "id",
        "node",
    ]
    
    node_id_col = next(
        (c for c in possible_node_id_cols if c in graph_nodes.columns),
        None
    )
    
    if node_id_col is None:
        raise ValueError(
            f"Cannot infer node_id column from graph_nodes: {graph_nodes.columns.tolist()}"
        )
    
    print("Using node_id_col:", node_id_col)
    
    # Add nodes
    for _, row in graph_nodes.iterrows():
        node_id = row[node_id_col]
        attrs = row.to_dict()
        G.add_node(node_id, **attrs)
    
    # Infer edge source/target columns
    possible_src_cols = [
        "source_node_id",
        "source",
        "source_id",
        "src",
        "from",
        "from_node_id",
    ]
    
    possible_dst_cols = [
        "target_node_id",
        "target",
        "target_id",
        "dst",
        "to",
        "to_node_id",
    ]
    
    src_col = next(
        (c for c in possible_src_cols if c in graph_edges.columns),
        None
    )
    
    dst_col = next(
        (c for c in possible_dst_cols if c in graph_edges.columns),
        None
    )
    
    if src_col is None or dst_col is None:
        raise ValueError(
            f"Cannot infer source/target columns from graph_edges: {graph_edges.columns.tolist()}"
        )
    
    print("Using src_col:", src_col)
    print("Using dst_col:", dst_col)
    
    # Add edges
    for _, row in graph_edges.iterrows():
        src = row[src_col]
        dst = row[dst_col]
        attrs = row.to_dict()
        G.add_edge(src, dst, **attrs)
    
    print("Graph created.")
    print("Graph nodes:", G.number_of_nodes())
    print("Graph edges:", G.number_of_edges())
    
    # Basic graph stats
    print("Is directed:", G.is_directed())
    print("Number of connected components:", nx.number_connected_components(G))


Using node_id_col: node_id
Using src_col: source_node_id
Using dst_col: target_node_id
Graph created.
Graph nodes: 1007
Graph edges: 4140
Is directed: False
Number of connected components: 1


### Simple graph features для команд

In [52]:
if GRAPH_NODES_PATH.exists() and GRAPH_EDGES_PATH.exists():
    degree_dict = dict(G.degree())
    degree_centrality_dict = nx.degree_centrality(G)
    
    print("graph_nodes columns:")
    print(graph_nodes.columns.tolist())
    
    # Infer columns
    possible_type_cols = [
        "node_type",
        "type",
        "label",
        "entity_type",
    ]
    
    possible_name_cols = [
        "name",
        "node_name",
        "team_name",
        "canonical_name",
        "display_name",
    ]
    
    type_col = next(
        (c for c in possible_type_cols if c in graph_nodes.columns),
        None
    )
    
    name_col = next(
        (c for c in possible_name_cols if c in graph_nodes.columns),
        None
    )
    
    if type_col is None:
        raise ValueError(
            f"Cannot infer node type column from graph_nodes: {graph_nodes.columns.tolist()}"
        )
    
    if name_col is None:
        raise ValueError(
            f"Cannot infer node name column from graph_nodes: {graph_nodes.columns.tolist()}"
        )
    
    print("Using type_col:", type_col)
    print("Using name_col:", name_col)
    print("Using node_id_col:", node_id_col)
    
    print("Unique node types:")
    print(graph_nodes[type_col].value_counts().head(20))
    
    # Team nodes: robust lowercase matching
    team_nodes = graph_nodes[
        graph_nodes[type_col].astype(str).str.lower().isin(
            ["team", "teams", "national_team", "country"]
        )
    ].copy()
    
    print("Team nodes:", team_nodes.shape)
    display(team_nodes.head())
    
    if len(team_nodes) == 0:
        raise ValueError(
            "No Team nodes found. Check node type values above and adjust team type filter."
        )
    
    team_name_to_node_id = dict(zip(team_nodes[name_col].astype(str), team_nodes[node_id_col]))
    
    # Optional: normalized mapping to reduce mismatch
    def normalize_team_name(x):
        if pd.isna(x):
            return ""
        return str(x).strip().lower()
    
    team_name_norm_to_node_id = {
        normalize_team_name(name): node_id
        for name, node_id in team_name_to_node_id.items()
    }
    
    def resolve_team_node_id(team_name):
        # Exact match
        if team_name in team_name_to_node_id:
            return team_name_to_node_id[team_name]
        
        # Normalized match
        norm = normalize_team_name(team_name)
        if norm in team_name_norm_to_node_id:
            return team_name_norm_to_node_id[norm]
        
        return None
    
    # Check match coverage
    all_match_teams = pd.concat([
        df_all["home_team_name"],
        df_all["away_team_name"]
    ]).dropna().astype(str).unique()
    
    unmatched = [
        t for t in all_match_teams
        if resolve_team_node_id(t) is None
    ]
    
    print(f"Unmatched teams: {len(unmatched)} / {len(all_match_teams)}")
    print(unmatched[:50])
    
    def add_simple_graph_features(df):
        df = df.copy()
        
        def get_team_degree(team_name):
            nid = resolve_team_node_id(team_name)
            if nid is None:
                return 0
            return degree_dict.get(nid, 0)
        
        def get_team_cent(team_name):
            nid = resolve_team_node_id(team_name)
            if nid is None:
                return 0.0
            return degree_centrality_dict.get(nid, 0.0)
        
        df["home_team_degree"] = df["home_team_name"].apply(get_team_degree)
        df["away_team_degree"] = df["away_team_name"].apply(get_team_degree)
        df["degree_diff"] = df["home_team_degree"] - df["away_team_degree"]
        
        df["home_team_centrality"] = df["home_team_name"].apply(get_team_cent)
        df["away_team_centrality"] = df["away_team_name"].apply(get_team_cent)
        df["centrality_diff"] = df["home_team_centrality"] - df["away_team_centrality"]
        
        return df
    
    train_graph_df = add_simple_graph_features(train_df)
    val_graph_df = add_simple_graph_features(val_df)
    test_graph_df = add_simple_graph_features(test_df)
    
    graph_feature_cols = [
        "home_team_degree",
        "away_team_degree",
        "degree_diff",
        "home_team_centrality",
        "away_team_centrality",
        "centrality_diff",
    ]
    
    graph_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    X_graph_train = graph_preprocessor.fit_transform(train_graph_df[graph_feature_cols])
    X_graph_val = graph_preprocessor.transform(val_graph_df[graph_feature_cols])
    X_graph_test = graph_preprocessor.transform(test_graph_df[graph_feature_cols])
    
    print("X_graph_train:", X_graph_train.shape)
    print("X_graph_val:", X_graph_val.shape)
    print("X_graph_test:", X_graph_test.shape)
    
    display(train_graph_df[
        ["home_team_name", "away_team_name"] + graph_feature_cols
    ].head())


graph_nodes columns:
['node_id', 'node_type', 'name', 'canonical_name', 'source_id', 'tournament_year', 'metadata_json']
Using type_col: node_type
Using name_col: name
Using node_id_col: node_id
Unique node types:
node_type
Match             500
TeamTournament    248
Stadium            93
Referee            88
Team               70
Tournament          8
Name: count, dtype: int64
Team nodes: (70, 7)


,node_id,node_type,name,canonical_name,source_id,tournament_year,metadata_json
0,team_28ef36b35ae8,Team,Germany,germany,NaN,NaN,"{""team_norm"": ""germany""}"
1,team_1747d4b2dd4b,Team,Spain,spain,NaN,NaN,"{""team_norm"": ""spain""}"
2,team_25446782e2cc,Team,Colombia,colombia,NaN,NaN,"{""team_norm"": ""colombia""}"
3,team_bab4ac375ada,Team,Italy,italy,NaN,NaN,"{""team_norm"": ""italy""}"
4,team_152649df347e,Team,United States,united states,NaN,NaN,"{""team_norm"": ""united states""}"


Unmatched teams: 0 / 70
[]
X_graph_train: (372, 6)
X_graph_val: (64, 6)
X_graph_test: (64, 6)


,home_team_name,away_team_name,home_team_degree,away_team_degree,degree_diff,home_team_centrality,away_team_centrality,centrality_diff
0,Germany,Bolivia,90,8,82,0.089463,0.007952,0.081511
1,Spain,Korea Republic,76,69,7,0.075547,0.068588,0.006958
2,Colombia,Romania,35,21,14,0.034791,0.020875,0.013917
3,Italy,Republic of Ireland,67,20,47,0.066600,0.019881,0.046720
4,United States,Switzerland,61,53,8,0.060636,0.052684,0.007952


### Graph-only и Numeric + Text + Graph baseline

In [54]:
if GRAPH_NODES_PATH.exists() and GRAPH_EDGES_PATH.exists() and "X_graph_train" in globals():
    fit_eval_register_model(
        model=make_logreg(),
        model_name="simple_graph_logreg",
        feature_set="simple_graph_features",
        X_train=X_graph_train,
        y_train=y_train,
        X_val=X_graph_val,
        y_val=y_val,
        X_test=X_graph_test,
        y_test=y_test,
        verbose=True
    )
    
    X_num_text_graph_train = np.hstack([X_fused_train, X_graph_train])
    X_num_text_graph_val = np.hstack([X_fused_val, X_graph_val])
    X_num_text_graph_test = np.hstack([X_fused_test, X_graph_test])
    
    fit_eval_register_model(
        model=make_logreg(),
        model_name="numeric_text_graph_logreg",
        feature_set="combined_numeric_text_simple_graph",
        X_train=X_num_text_graph_train,
        y_train=y_train,
        X_val=X_num_text_graph_val,
        y_val=y_val,
        X_test=X_num_text_graph_test,
        y_test=y_test,
        verbose=True
    )
    
    fit_eval_register_model(
        model=make_mlp_safe(),
        model_name="numeric_text_graph_mlp",
        feature_set="combined_numeric_text_simple_graph",
        X_train=X_num_text_graph_train,
        y_train=y_train,
        X_val=X_num_text_graph_val,
        y_val=y_val,
        X_test=X_num_text_graph_test,
        y_test=y_test,
        verbose=True
    )
    
    results_df = get_results_df()
    results_df.to_csv(RESULTS_PATH, index=False)
    display(results_df)



simple_graph_logreg | VAL
Accuracy: 0.40625
Balanced accuracy: 0.36564102564102563
Macro F1: 0.3646153846153846
Weighted F1: 0.40531249999999996
Draw recall: 0.15384615384615385
Draw F1: 0.15384615384615385

Confusion matrix:
[[11  6  9]
 [ 6  2  5]
 [ 7  5 13]]

Classification report:
              precision    recall  f1-score   support

    home_win       0.46      0.42      0.44        26
        draw       0.15      0.15      0.15        13
    away_win       0.48      0.52      0.50        25

    accuracy                           0.41        64
   macro avg       0.36      0.37      0.36        64
weighted avg       0.41      0.41      0.41        64


simple_graph_logreg | TEST
Accuracy: 0.53125
Balanced accuracy: 0.4850574712643678
Macro F1: 0.47804232804232805
Weighted F1: 0.5262276785714286
Draw recall: 0.2
Draw F1: 0.2222222222222222

Confusion matrix:
[[19  5  5]
 [ 4  3  8]
 [ 4  4 12]]

Classification report:
              precision    recall  f1-score   support

    h

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
2,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
22,numeric_text_rf,combined_numeric_plus_text_embeddings,0.640625,0.562051,0.546268,0.606456,0.153846,0.235294,0.578125,0.551724,0.553846,0.576923,0.400000,0.461538
6,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
4,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
7,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097
10,xgboost_combined_numeric,combined_numeric,0.515625,0.469231,0.479167,0.536133,0.230769,0.187500,0.562500,0.529885,0.530586,0.560436,0.400000,0.413793
9,combined_numeric_mlp_safe,combined_numeric,0.578125,0.497436,0.470588,0.539216,0.076923,0.117647,0.406250,0.366475,0.342767,0.387824,0.066667,0.100000
3,ranking_full_logreg,ranking_full,0.531250,0.471795,0.463509,0.522321,0.153846,0.160000,0.484375,0.472031,0.467174,0.499899,0.333333,0.303030
8,combined_numeric_histgb_safe,combined_numeric,0.562500,0.470769,0.452991,0.541266,0.000000,0.000000,0.562500,0.513218,0.501771,0.543780,0.200000,0.260870
1,ranking_compact_logreg,ranking_compact,0.484375,0.443077,0.451558,0.503321,0.230769,0.187500,0.515625,0.495402,0.493690,0.522967,0.400000,0.363636


# Optional Node2Vec baseline

### Node2Vec installation check

In [57]:
try:
    from node2vec import Node2Vec
    HAS_NODE2VEC = True
    print("node2vec available.")
except ImportError:
    HAS_NODE2VEC = False
    print("node2vec not installed. Install with:")
    print("!pip install node2vec")


node2vec not installed. Install with:
!pip install node2vec
